In [1]:
import torch
from training_loop import training_loop
from extract_embedding import get_train_test_loaders
from qwen_tts.core.models import BasicSpeakerEncoder
import optuna

/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



********
********
 


2026-04-22 03:39:22.998652675 [W:onnxruntime:Default, device_discovery.cc:164 DiscoverDevicesForPlatform] GPU device discovery failed: device_discovery.cc:89 ReadFileContents Failed to open file: "/sys/class/drm/card0/device/vendor"


In [2]:
# Setup the data loaders
train_loader, test_loader = get_train_test_loaders()


Device: cuda
GPU: NVIDIA GeForce RTX 4060 Laptop GPU
VRAM: 8.6 GB
Loading Qwen3-TTS model...


Fetching 4 files: 100%|██████████| 4/4 [00:00<00:00, 32078.81it/s]


Qwen3 speaker encoder loaded on cuda
train-clean-100 already extracted
dev-clean already extracted
Loaded 12526 samples
Loaded 1953 samples
Train: 12526 samples, 391 batches
Test:  1953 samples, 62 batches


In [3]:

def objective(trial, train_loader, test_loader):
    # 1. Define the hyperparameter search space
    # Optuna will sample values from these ranges for each trial
    lr = trial.suggest_float("lr", 1e-5, 1e-2, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True)
    alpha = trial.suggest_float("loss_alpha", 0.1, 0.9) 
    
    # Assemble the dictionary for the training loop
    hyperparams = {
        "num_epochs": 20,       # Keep epochs lower for faster sweeps (tune as needed)
        "save_every": 20,       # Prevent excessive checkpoint saving during sweeps
        "lr": lr,
        "loss_alpha": alpha,
        "weight_decay": weight_decay,
        "model_name": f"BasicSpeakerEncoder_trial_{trial.number}"
    }

    # 2. Setup Device and model
    if torch.cuda.is_available():
        device = torch.device("cuda")
    elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        device = torch.device("mps")
    else:
        device = torch.device("cpu")

    # Setup the model
    model = BasicSpeakerEncoder().to(device)
    
        
    # 3. Execute the training loop
    # We catch the returned best test cosine similarity
    best_test_cosine = training_loop(model, train_loader, test_loader, device, hyperparams)
    
    # Optuna will try to MAXIMIZE this returned value
    return best_test_cosine


In [4]:
study = optuna.create_study(direction="minimize", study_name="Speaker_Encoder_Distillation")
study.optimize(lambda trial: objective(trial, train_loader, test_loader), n_trials=20)
    
print("\n=================================")
print("Best Trial Found:")
print(f"Best Test Loss: {study.best_trial.value:.4f}")
print("Optimal Hyperparameters: ")
for key, value in study.best_trial.params.items():
    print(f"{key}: {value}")

[I 2026-04-22 03:39:41,479] A new study created in memory with name: Speaker_Encoder_Distillation


Epoch   1/20 | Train loss=0.0648 cos=0.9569 | Test loss=0.0517 cos=0.9744 *best* | 8.5s
Epoch   2/20 | Train loss=0.0471 cos=0.9805 | Test loss=0.0488 cos=0.9783 *best* | 7.6s
Epoch   3/20 | Train loss=0.0444 cos=0.9841 | Test loss=0.0454 cos=0.9827 *best* | 7.7s
Epoch   4/20 | Train loss=0.0430 cos=0.9858 | Test loss=0.0448 cos=0.9835 *best* | 7.9s
Epoch   5/20 | Train loss=0.0423 cos=0.9868 | Test loss=0.0447 cos=0.9837 *best* | 7.8s
Epoch   6/20 | Train loss=0.0418 cos=0.9875 | Test loss=0.0442 cos=0.9843 *best* | 7.7s
Epoch   7/20 | Train loss=0.0413 cos=0.9881 | Test loss=0.0441 cos=0.9845 *best* | 7.8s
Epoch   8/20 | Train loss=0.0410 cos=0.9886 | Test loss=0.0441 cos=0.9844 | 7.9s
Epoch   9/20 | Train loss=0.0407 cos=0.9890 | Test loss=0.0436 cos=0.9852 *best* | 7.9s
Epoch  10/20 | Train loss=0.0404 cos=0.9894 | Test loss=0.0433 cos=0.9855 *best* | 7.7s
Epoch  11/20 | Train loss=0.0402 cos=0.9897 | Test loss=0.0431 cos=0.9859 *best* | 7.8s
Epoch  12/20 | Train loss=0.0399 cos=0.

[I 2026-04-22 03:42:19,201] Trial 0 finished with value: 0.986949170789411 and parameters: {'lr': 0.004965301068512949, 'weight_decay': 1.5972828920435673e-06, 'loss_alpha': 0.7488432322919527}. Best is trial 0 with value: 0.986949170789411.


Epoch  20/20 | Train loss=0.0387 cos=0.9916 | Test loss=0.0422 cos=0.9869 *best* | 8.0s

Done! Best test cosine similarity: 0.9869
Epoch   1/20 | Train loss=0.1349 cos=0.8622 | Test loss=0.0742 cos=0.9693 *best* | 7.9s
Epoch   2/20 | Train loss=0.0706 cos=0.9756 | Test loss=0.0711 cos=0.9748 *best* | 8.0s
Epoch   3/20 | Train loss=0.0687 cos=0.9790 | Test loss=0.0698 cos=0.9771 *best* | 8.1s
Epoch   4/20 | Train loss=0.0676 cos=0.9808 | Test loss=0.0687 cos=0.9790 *best* | 8.2s
Epoch   5/20 | Train loss=0.0669 cos=0.9821 | Test loss=0.0681 cos=0.9801 *best* | 7.9s
Epoch   6/20 | Train loss=0.0664 cos=0.9830 | Test loss=0.0677 cos=0.9808 *best* | 8.1s
Epoch   7/20 | Train loss=0.0660 cos=0.9837 | Test loss=0.0675 cos=0.9812 *best* | 8.2s
Epoch   8/20 | Train loss=0.0657 cos=0.9843 | Test loss=0.0671 cos=0.9819 *best* | 8.2s
Epoch   9/20 | Train loss=0.0654 cos=0.9847 | Test loss=0.0668 cos=0.9824 *best* | 8.0s
Epoch  10/20 | Train loss=0.0652 cos=0.9851 | Test loss=0.0667 cos=0.9825 *be

[I 2026-04-22 03:45:01,294] Trial 1 finished with value: 0.9839857941673648 and parameters: {'lr': 5.1260841604696016e-05, 'weight_decay': 4.3304603410571115e-06, 'loss_alpha': 0.559769724443428}. Best is trial 1 with value: 0.9839857941673648.


Epoch  20/20 | Train loss=0.0643 cos=0.9867 | Test loss=0.0659 cos=0.9839 | 7.9s

Done! Best test cosine similarity: 0.9840
Epoch   1/20 | Train loss=0.0952 cos=0.9329 | Test loss=0.0716 cos=0.9752 *best* | 8.0s
Epoch   2/20 | Train loss=0.0686 cos=0.9803 | Test loss=0.0690 cos=0.9798 *best* | 8.1s
Epoch   3/20 | Train loss=0.0670 cos=0.9831 | Test loss=0.0680 cos=0.9815 *best* | 8.0s
Epoch   4/20 | Train loss=0.0662 cos=0.9847 | Test loss=0.0673 cos=0.9828 *best* | 8.0s


[W 2026-04-22 03:45:38,973] Trial 2 failed with parameters: {'lr': 0.0002383924101793105, 'weight_decay': 8.120583115757078e-05, 'loss_alpha': 0.5531739004027564} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/usr/local/lib/python3.10/dist-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
  File "/tmp/ipykernel_89336/2882467666.py", line 2, in <lambda>
    study.optimize(lambda trial: objective(trial, train_loader, test_loader), n_trials=20)
  File "/tmp/ipykernel_89336/2337655088.py", line 32, in objective
    best_test_cosine = training_loop(model, train_loader, test_loader, device, hyperparams)
  File "/workspace/training_loop.py", line 44, in training_loop
    epoch_loss += loss.item()
KeyboardInterrupt
[W 2026-04-22 03:45:38,975] Trial 2 failed with value None.


KeyboardInterrupt: 